# Spanish fine-tuning + official scoring

This notebook is designed for:
1. Fine-tune **Spanish-specific XLM-R** and evaluate on Spanish (`ES -> ES`).
2. Optimize post-processing on a Spanish dev split for stronger official **IoU/Correlation**.
3. Compare against a fixed EN->ES baseline score row from your English notebook.
4. Score with the official Mu-SHROOM `participant_kit/scorer.py`.

In [1]:
# Colab setup
!pip install -q "datasets>=2.18.0" "transformers>=4.41.0" "accelerate>=0.29.0" evaluate seqeval scikit-learn pandas

import os
import sys
import json
import math
import inspect
import random
import subprocess
import importlib.util
from pathlib import Path
from typing import Dict, List, Optional, Tuple

from google.colab import drive
drive.mount("/content/drive")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import Dataset, DatasetDict, load_dataset
from sklearn.metrics import precision_recall_fscore_support, f1_score
from sklearn.utils.class_weight import compute_class_weight
from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    DataCollatorForTokenClassification,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
    set_seed,
)
import evaluate

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEQEVAL = evaluate.load("seqeval")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 534.9 kB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 1.2 MB/s eta 0:00:00
Mounted at /content/drive
torch: 2.10.0+cpu
cuda available: False


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [10]:
# Configuration for Spanish and English
MODEL_NAME = "xlm-roberta-large"
LANGUAGE = "es" ##language code for spanish in mushroom
SEED = 4650
MAX_LENGTH = 512

##might need to add english checkpoint but will double check

ENGLISH_CHECKPOINT  = "/content/drive/MyDrive/xlmr-mu-shroom-en-best"

OUTPUT_ROOT = "/content/mu-shroom-hi-xlmr-large"
SPANISH_CHECKPOINT_DIR = os.path.join(OUTPUT_ROOT, "best")
PREDICTIONS_DIR = os.path.join(OUTPUT_ROOT, "predictions")
SCORER_REPO_DIR = "/content/mu-shroom"

RUN_SPANISH_TRAINING = True
TRAIN_BATCH_SIZE = 1
EVAL_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 8
##will double check number of epochs and change later
NUM_EPOCHS = 10
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
EARLY_STOPPING_PATIENCE = 2

PREDICTION_THRESHOLD = 0.50
MERGE_GAP = 1
MIN_SPAN_LEN = 1

label_list = ["O", "B-HALL", "I-HALL"]
label2id = {label: idx for idx, label in enumerate(label_list)}
id2label = {idx: label for label, idx in label2id.items()}

os.makedirs(OUTPUT_ROOT, exist_ok=True)
os.makedirs(PREDICTIONS_DIR, exist_ok=True)
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
print("Language:", LANGUAGE)
print("Output root:", OUTPUT_ROOT)
print()
#print("Hardcoded inference params:", PREDICTION_THRESHOLD, PREDICTION_TEMPERATURE, MERGE_GAP, MIN_SPAN_LEN)

Language: es
Output root: /content/mu-shroom-hi-xlmr-large


## Loading the Spanish Mu-SHROOM Dataset

This cell should load the Spanish subset and should confirm which columns contain:
- the generated answer text,
- the hard hallucination spans,
- the soft per-span probabilities used by the official scorer.

We use schema inference to ensure reliable workability even if the dataset config exposes the same fields through slightly different split layouts.

In [11]:
# Load dataset and test sllit
def load_mu_shroom_language(language):
    try:
        ds = load_dataset("Helsinki-NLP/mu-shroom", language)
        print(f"Loaded config: {language}")
        return ds
    except Exception as e:
        print(f"Direct load failed: {e}")
        ds = load_dataset("Helsinki-NLP/mu-shroom", "all")
        for col in ["language", "lang", "locale"]:
            if col in ds[next(iter(ds.keys()))].column_names:
                return DatasetDict({
                    split: ds[split].filter(lambda r: r[col] == language)
                    for split in ds
                })
        raise RuntimeError("Could not filter by language.")


def infer_schema(example_row):
    columns = list(example_row.keys())
    def choose(candidates):
        for c in candidates:
            if c in columns:
                return c
        return None
    return {
        "text_col": choose(["model_output_text", "answer", "output_text", "text", "response"]),
        "hard_col": choose(["hard_labels", "hard_spans", "hallucination_spans"]),
        "soft_col": choose(["soft_labels", "soft_spans", "probabilistic_labels"]),
        "id_col":   choose(["id", "sample_id", "instance_id"]),
    }


raw_ds  = load_mu_shroom_language(LANGUAGE)
preview_split = "validation" if "validation" in raw_ds else next(iter(raw_ds.keys()))
schema   = infer_schema(raw_ds[preview_split][0])
text_col = schema["text_col"]
hard_col = schema["hard_col"]
id_col = schema["id_col"] or "id"

print("Schema:", schema)
if text_col is None or hard_col is None:
    raise ValueError("Could not infer text/hard label columns.")

# Spanish-specific: train_unlabeled has no labels so we ignore it.
# Use validation (50 rows) for train/dev, test (152 rows) as final eval.
print("\nChecking label coverage:")
print("validation hard_labels sample:", raw_ds["validation"][0]["hard_labels"])
print("test hard_labels sample:      ", raw_ds["test"][0]["hard_labels"])

val_split = raw_ds["validation"].train_test_split(test_size=0.15, seed=SEED)
train_raw = val_split["train"]   # ~42 rows
dev_raw = val_split["test"]    # ~8 rows
final_eval_raw = raw_ds["test"]       # 152 rows, held out entirely

# Verify no leakage
train_ids = set(train_raw[id_col]) if id_col in train_raw.column_names else set(range(len(train_raw)))
dev_ids = set(dev_raw[id_col])   if id_col in dev_raw.column_names   else set(range(len(dev_raw)))
eval_ids = set(final_eval_raw[id_col]) if id_col in final_eval_raw.column_names else set(range(len(final_eval_raw)))

print(f"\ntrain: {len(train_raw)} | dev: {len(dev_raw)} | eval: {len(final_eval_raw)}")
print(f"overlap(train, dev):  {len(train_ids & dev_ids)}")
print(f"overlap(train, eval): {len(train_ids & eval_ids)}")
print(f"overlap(dev,   eval): {len(dev_ids   & eval_ids)}")

Loaded config: es
Schema: {'text_col': 'model_output_text', 'hard_col': 'hard_labels', 'soft_col': 'soft_labels', 'id_col': 'id'}

Checking label coverage:
validation hard_labels sample: [[56, 62], [167, 179]]
test hard_labels sample:       [[69, 105]]

train: 42 | dev: 8 | eval: 152
overlap(train, dev):  0
overlap(train, eval): 0
overlap(dev,   eval): 0


In [4]:
##Helper Functions

def json_default(obj):
    if hasattr(obj, "item"):
        return obj.item()
    if isinstance(obj, set):
        return list(obj)
    raise TypeError(f"Type not serializable: {type(obj)}")


def normalize_hard_spans(raw_spans) -> List[List[int]]:
    if raw_spans is None:
        return []

    normalized = []
    for span in raw_spans:
        if span is None:
            continue
        if isinstance(span, dict):
            start = span.get("start")
            end = span.get("end")
        elif isinstance(span, (list, tuple)) and len(span) >= 2:
            start, end = span[0], span[1]
        else:
            continue

        if start is None or end is None:
            continue
        start = int(start)
        end = int(end)
        if end > start:
            normalized.append([start, end])

    normalized.sort(key=lambda pair: (pair[0], pair[1]))
    return normalized


def token_overlaps_span(token_start: int, token_end: int, span_start: int, span_end: int) -> bool:
    return (token_start < span_end) and (token_end > span_start)


def spans_to_token_bio(offset_mapping, raw_spans, label2id_map) -> List[int]:
    spans = normalize_hard_spans(raw_spans)
    labels = []
    active_span_index = None

    for token_start, token_end in offset_mapping:
        token_start = int(token_start)
        token_end = int(token_end)

        if token_start == token_end == 0:
            labels.append(-100)
            continue

        matched_index = None
        for span_index, (span_start, span_end) in enumerate(spans):
            if token_overlaps_span(token_start, token_end, span_start, span_end):
                matched_index = span_index
                break

        if matched_index is None:
            labels.append(label2id_map["O"])
            active_span_index = None
            continue

        span_start, _ = spans[matched_index]
        begins_here = (
            token_start <= span_start < token_end
            or token_start == span_start
            or active_span_index != matched_index
        )
        labels.append(label2id_map["B-HALL"] if begins_here else label2id_map["I-HALL"])
        active_span_index = matched_index

    previous_inside = False
    for idx, label_id in enumerate(labels):
        if label_id in (-100, label2id_map["O"]):
            previous_inside = False
            continue
        if label_id == label2id_map["I-HALL"] and not previous_inside:
            labels[idx] = label2id_map["B-HALL"]
        previous_inside = True
    return labels


def merge_spans(spans: List[List[int]], gap: int = 0) -> List[List[int]]:
    spans = sorted(normalize_hard_spans(spans), key=lambda pair: (pair[0], pair[1]))
    if not spans:
        return []

    merged = [spans[0][:]]
    for start, end in spans[1:]:
        prev_start, prev_end = merged[-1]
        if start <= prev_end + gap:
            merged[-1][1] = max(prev_end, end)
        else:
            merged.append([start, end])
    return merged


def token_labels_to_char_spans(offset_mapping, pred_ids, id2label_map, merge_gap: int = 0) -> List[List[int]]:
    spans = []
    current_start = None
    current_end = None

    for (token_start, token_end), pred_id in zip(offset_mapping, pred_ids):
        token_start = int(token_start)
        token_end = int(token_end)
        if token_start == token_end == 0:
            continue

        label_name = id2label_map[int(pred_id)]
        if label_name == "B-HALL":
            if current_start is not None:
                spans.append([current_start, current_end])
            current_start, current_end = token_start, token_end
        elif label_name == "I-HALL" and current_start is not None:
            current_end = token_end
        else:
            if current_start is not None:
                spans.append([current_start, current_end])
                current_start, current_end = None, None

    if current_start is not None:
        spans.append([current_start, current_end])

    return merge_spans(spans, gap=merge_gap)


def token_probs_to_soft_labels(offset_mapping, hall_probs: List[float]) -> List[Dict[str, float]]:
    soft_labels = []
    for (token_start, token_end), prob in zip(offset_mapping, hall_probs):
        token_start = int(token_start)
        token_end = int(token_end)
        if token_start >= token_end:
            continue
        soft_labels.append({"start": token_start, "end": token_end, "prob": float(prob)})
    return soft_labels


def preprocess_batch(batch, tokenizer_obj, text_column: str, hard_column: str, max_length: int = 512):
    encodings = tokenizer_obj(
        batch[text_column],
        truncation=True,
        max_length=max_length,
        return_offsets_mapping=True,
    )
    encodings["labels"] = [
        spans_to_token_bio(offsets, raw_spans, label2id)
        for offsets, raw_spans in zip(encodings["offset_mapping"], batch[hard_column])
    ]
    return encodings


def flatten_non_ignored(labels: List[List[int]]) -> np.ndarray:
    return np.array([label for row in labels for label in row if label != -100], dtype=np.int64)


def compute_class_weights(train_dataset, num_labels: int) -> torch.Tensor:
    y = flatten_non_ignored(train_dataset["labels"])
    classes = np.arange(num_labels)
    weights = compute_class_weight(class_weight="balanced", classes=classes, y=y)
    return torch.tensor(weights, dtype=torch.float32)


def compute_token_metrics(eval_pred):
    logits, labels = eval_pred
    pred_ids = np.argmax(logits, axis=-1)

    seqeval_predictions = []
    seqeval_references = []
    flat_pred = []
    flat_gold = []

    for pred_row, gold_row in zip(pred_ids, labels):
        filtered_pred = []
        filtered_gold = []
        for pred_id, gold_id in zip(pred_row, gold_row):
            if gold_id == -100:
                continue
            pred_name = id2label[int(pred_id)]
            gold_name = id2label[int(gold_id)]
            filtered_pred.append(pred_name)
            filtered_gold.append(gold_name)
            flat_pred.append(int(pred_id))
            flat_gold.append(int(gold_id))
        seqeval_predictions.append(filtered_pred)
        seqeval_references.append(filtered_gold)

    precision, recall, f1_micro, _ = precision_recall_fscore_support(
        flat_gold,
        flat_pred,
        labels=[label2id["B-HALL"], label2id["I-HALL"]],
        average="micro",
        zero_division=0,
    )
    f1_macro = f1_score(flat_gold, flat_pred, average="macro", zero_division=0)
    f1_hall_mean = (
        f1_score(flat_gold, flat_pred, labels=[label2id["B-HALL"]], average="macro", zero_division=0)
        + f1_score(flat_gold, flat_pred, labels=[label2id["I-HALL"]], average="macro", zero_division=0)
    ) / 2.0

    metrics = {
        "token_precision_hall": float(precision),
        "token_recall_hall": float(recall),
        "token_f1_hall_micro": float(f1_micro),
        "token_f1_macro": float(f1_macro),
        "f1_hall_mean": float(f1_hall_mean),
    }

    try:
        seqeval_metrics = SEQEVAL.compute(
            predictions=seqeval_predictions,
            references=seqeval_references,
            zero_division=0,
        )
    except TypeError:
        seqeval_metrics = SEQEVAL.compute(
            predictions=seqeval_predictions,
            references=seqeval_references,
        )
    metrics.update(seqeval_metrics)
    return metrics


class WeightedTrainer(Trainer):
  def __init__(self, class_weights: torch.Tensor, *args, **kwargs):
      super().__init__(*args, **kwargs)
      self._ce_weights = class_weights.detach().clone()

  def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
      labels = inputs["labels"]
      outputs = model(**inputs)
      logits = outputs.logits
      loss_fn = nn.CrossEntropyLoss(
          weight=self._ce_weights.to(device=logits.device, dtype=logits.dtype),
          ignore_index=-100,
      )
      loss = loss_fn(logits.view(-1, logits.shape[-1]), labels.view(-1))
      return (loss, outputs) if return_outputs else loss


def make_training_args(output_dir: str) -> TrainingArguments:
    signature = inspect.signature(TrainingArguments.__init__)
    params = signature.parameters
    kwargs = dict(
        output_dir=output_dir,
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        num_train_epochs=NUM_EPOCHS,
        weight_decay=WEIGHT_DECAY,
        warmup_ratio=WARMUP_RATIO,
        logging_steps=10,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="f1_hall_mean",
        greater_is_better=True,
        fp16=torch.cuda.is_available(),
        seed=SEED,
        report_to="none",
    )
    if "evaluation_strategy" in params:
        kwargs["evaluation_strategy"] = "epoch"
        kwargs["save_strategy"] = "epoch"
    else:
        if "eval_strategy" in params:
            kwargs["eval_strategy"] = "epoch"
        if "save_strategy" in params:
            kwargs["save_strategy"] = "epoch"
    return TrainingArguments(**kwargs)


def write_jsonl(path: str, rows: List[Dict]):
    with open(path, "w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=False, default=json_default) + "\n")


def clone_if_missing(url: str, path: str):
    if os.path.isdir(path):
        print(f"Using existing directory: {path}")
        return
    subprocess.check_call(["git", "clone", "--depth", "1", url, path])


def load_scorer_module(repo_dir: str):
    scorer_path = os.path.join(repo_dir, "participant_kit", "scorer.py")
    if not os.path.isfile(scorer_path):
        raise FileNotFoundError(
            f"Expected scorer.py at {scorer_path}. If you already cloned the repo elsewhere, update SCORER_REPO_DIR."
        )
    spec = importlib.util.spec_from_file_location("mu_shroom_scorer", scorer_path)
    module = importlib.util.module_from_spec(spec)
    assert spec.loader is not None
    spec.loader.exec_module(module)
    return module


def build_reference_jsonl(split_ds: Dataset, schema_map: Dict[str, Optional[str]], out_path: str):
    rows = []
    text_col = schema_map["text_col"]
    hard_col = schema_map["hard_col"]
    soft_col = schema_map["soft_col"]
    id_col = schema_map["id_col"] or "id"

    for row_index, row in enumerate(split_ds):
        rows.append(
            {
                "id": row.get(id_col, row_index),
                "model_output_text": row[text_col],
                "soft_labels": row.get(soft_col, []),
                "hard_labels": normalize_hard_spans(row.get(hard_col, [])),
            }
        )
    write_jsonl(out_path, rows)
    return out_path


def score_predictions_with_official_scorer(reference_jsonl: str, prediction_jsonl: str, score_output_txt: str):
    clone_if_missing("https://github.com/Helsinki-NLP/mu-shroom", SCORER_REPO_DIR)
    scorer = load_scorer_module(SCORER_REPO_DIR)
    reference_records = scorer.load_jsonl_file_to_records(reference_jsonl, is_ref=True)
    prediction_records = scorer.load_jsonl_file_to_records(prediction_jsonl, is_ref=False)
    ious, correlations = scorer.main(reference_records, prediction_records, score_output_txt)
    return float(np.mean(ious)), float(np.mean(correlations))


def predict_dataset_to_official_rows(
    checkpoint_dir: str,
    dataset_split: Dataset,
    schema_map: Dict[str, Optional[str]],
    threshold: float = 0.5,
    merge_gap: int = 0,
    tokenizer_name: Optional[str] = None,
):
    checkpoint_tokenizer = AutoTokenizer.from_pretrained(tokenizer_name or checkpoint_dir, use_fast=True)
    checkpoint_model = AutoModelForTokenClassification.from_pretrained(checkpoint_dir)
    checkpoint_model.to(DEVICE)
    checkpoint_model.eval()

    text_col = schema_map["text_col"]
    id_col = schema_map["id_col"] or "id"
    checkpoint_id2label = {int(key): value for key, value in checkpoint_model.config.id2label.items()}
    hall_ids = [idx for idx, name in checkpoint_id2label.items() if name in {"B-HALL", "I-HALL"}]

    rows = []
    token_metric_gold = []
    token_metric_pred = []

    with torch.no_grad():
        for row_index, row in enumerate(dataset_split):
            text = row[text_col]
            encoded = checkpoint_tokenizer(
                text,
                return_offsets_mapping=True,
                return_tensors="pt",
                truncation=True,
                max_length=MAX_LENGTH,
            )
            offset_mapping = encoded.pop("offset_mapping")[0].tolist()
            encoded = {key: value.to(DEVICE) for key, value in encoded.items()}

            logits = checkpoint_model(**encoded).logits[0]
            probabilities = F.softmax(logits, dim=-1)
            pred_ids = logits.argmax(dim=-1).tolist()

            hall_probs = []
            for token_index in range(probabilities.shape[0]):
                if hall_ids:
                    hall_prob = float(probabilities[token_index, hall_ids].sum().item())
                else:
                    hall_prob = 0.0
                hall_probs.append(hall_prob)

            hard_spans = token_labels_to_char_spans(offset_mapping, pred_ids, checkpoint_id2label, merge_gap=merge_gap)
            if threshold > 0.0:
                thresholded_spans = []
                for (start, end), prob, pred_id in zip(offset_mapping, hall_probs, pred_ids):
                    start = int(start)
                    end = int(end)
                    if start >= end:
                        continue
                    if prob >= threshold and checkpoint_id2label[int(pred_id)] in {"B-HALL", "I-HALL"}:
                        thresholded_spans.append([start, end])
                hard_spans = merge_spans(thresholded_spans, gap=merge_gap)

            rows.append(
                {
                    "id": row.get(id_col, row_index),
                    "hard_labels": hard_spans,
                    "soft_labels": token_probs_to_soft_labels(offset_mapping, hall_probs),
                }
            )

            if schema_map["hard_col"] is not None:
                gold_labels = spans_to_token_bio(offset_mapping, row[schema_map["hard_col"]], label2id)
                for gold_label, pred_id, (start, end) in zip(gold_labels, pred_ids, offset_mapping):
                    if int(start) == int(end) == 0 or gold_label == -100:
                        continue
                    token_metric_gold.append(gold_label)
                    pred_name = checkpoint_id2label[int(pred_id)]
                    token_metric_pred.append(label2id.get(pred_name, label2id["O"]))

    token_metrics = {}
    if token_metric_gold:
        precision, recall, f1_hall, _ = precision_recall_fscore_support(
            token_metric_gold,
            token_metric_pred,
            labels=[label2id["B-HALL"], label2id["I-HALL"]],
            average="micro",
            zero_division=0,
        )
        token_metrics = {
            "token_precision_hall": float(precision),
            "token_recall_hall": float(recall),
            "token_f1_hall_micro": float(f1_hall),
            "token_f1_macro": float(f1_score(token_metric_gold, token_metric_pred, average="macro", zero_division=0)),
        }
    return rows, token_metrics

In [5]:
train_tok = train_raw.map(
    lambda batch: preprocess_batch(batch, tokenizer, text_column=text_col, hard_column=hard_col),
    batched=True, remove_columns=train_raw.column_names,
)
dev_tok = dev_raw.map(
    lambda batch: preprocess_batch(batch, tokenizer, text_column=text_col, hard_column=hard_col),
    batched=True, remove_columns=dev_raw.column_names,
)

class_weights = compute_class_weights(train_tok, len(label_list))
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)
print("Class weights [O, B-HALL, I-HALL]:", class_weights.tolist())

Map:   0%|          | 0/42 [00:00<?, ? examples/s]

Map:   0%|          | 0/8 [00:00<?, ? examples/s]

Class weights [O, B-HALL, I-HALL]: [0.38659563660621643, 14.812925338745117, 2.891766309738159]


#Run Training for Spanish model for Spanish Checkpoint

In [7]:
if RUN_SPANISH_TRAINING:
    spanish_model = AutoModelForTokenClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(label_list),
        id2label=id2label,
        label2id=label2id,
    )

    training_args = make_training_args(OUTPUT_ROOT)
    trainer_kwargs = dict(
        class_weights=class_weights,
        model=spanish_model,
        args=training_args,
        train_dataset=train_tok,
        eval_dataset=dev_tok,
        data_collator=data_collator,
        compute_metrics=compute_token_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
    )
    try:
        trainer = WeightedTrainer(processing_class=tokenizer, **trainer_kwargs)
    except TypeError:
        trainer = WeightedTrainer(tokenizer=tokenizer, **trainer_kwargs)

    train_result = trainer.train()
    print("Training finished.")
    print(train_result)

    os.makedirs(SPANISH_CHECKPOINT_DIR, exist_ok=True)
    trainer.model.save_pretrained(SPANISH_CHECKPOINT_DIR, safe_serialization=True)
    tokenizer.save_pretrained(SPANISH_CHECKPOINT_DIR)
    print("Saved Spanish checkpoint to:", SPANISH_CHECKPOINT_DIR)
else:
    print("Skipping Spanish fine-tuning. Using existing checkpoint path:", SPANISH_CHECKPOINT_DIR)

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

XLMRobertaForTokenClassification LOAD REPORT from: xlm-roberta-large
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
classifier.weight           | MISSING    | 
classifier.bias             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,Token Precision Hall,Token Recall Hall,Token F1 Hall Micro,Token F1 Macro,F1 Hall Mean,Hall,Overall Precision,Overall Recall,Overall F1,Overall Accuracy
1,No log,0.900698,0.000000,0.000000,0.000000,0.323829,0.000000,"{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 13}",0.000000,0.000000,0.000000,0.943322
2,8.054290,0.820806,0.044534,0.207547,0.073333,0.331936,0.042308,"{'precision': 0.02834008097165992, 'recall': 0.5384615384615384, 'f1': 0.05384615384615384, 'number': 13}",0.028340,0.538462,0.053846,0.829316
3,8.054290,0.709650,0.110000,0.207547,0.143791,0.401789,0.121711,"{'precision': 0.07368421052631578, 'recall': 0.5384615384615384, 'f1': 0.12962962962962962, 'number': 13}",0.073684,0.538462,0.129630,0.921173
4,5.462641,0.739394,0.106383,0.377358,0.165975,0.430747,0.179878,"{'precision': 0.051094890510948905, 'recall': 0.5384615384615384, 'f1': 0.09333333333333332, 'number': 13}",0.051095,0.538462,0.093333,0.872313
5,4.054197,0.647636,0.151163,0.245283,0.187050,0.448900,0.190920,"{'precision': 0.10144927536231885, 'recall': 0.5384615384615384, 'f1': 0.17073170731707318, 'number': 13}",0.101449,0.538462,0.170732,0.929642
6,4.054197,0.686612,0.125000,0.301887,0.176796,0.447787,0.195391,"{'precision': 0.08791208791208792, 'recall': 0.6153846153846154, 'f1': 0.15384615384615385, 'number': 13}",0.087912,0.615385,0.153846,0.906840
7,3.284000,0.711257,0.113636,0.283019,0.162162,0.443850,0.190212,"{'precision': 0.07865168539325842, 'recall': 0.5384615384615384, 'f1': 0.1372549019607843, 'number': 13}",0.078652,0.538462,0.137255,0.903583
8,3.284000,0.682832,0.160920,0.264151,0.200000,0.471828,0.224806,"{'precision': 0.09230769230769231, 'recall': 0.46153846153846156, 'f1': 0.15384615384615385, 'number': 13}",0.092308,0.461538,0.153846,0.930945
9,2.531932,0.683414,0.170732,0.264151,0.207407,0.478089,0.233318,"{'precision': 0.1, 'recall': 0.46153846153846156, 'f1': 0.16438356164383564, 'number': 13}",0.100000,0.461538,0.164384,0.934202
10,2.152860,0.693196,0.150538,0.264151,0.191781,0.472095,0.226268,"{'precision': 0.09090909090909091, 'recall': 0.46153846153846156, 'f1': 0.1518987341772152, 'number': 13}",0.090909,0.461538,0.151899,0.927036


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Training finished.
TrainOutput(global_step=60, training_loss=4.256653277079264, metrics={'train_runtime': 985.2595, 'train_samples_per_second': 0.426, 'train_steps_per_second': 0.061, 'total_flos': 80518443663660.0, 'train_loss': 4.256653277079264, 'epoch': 10.0})


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved Spanish checkpoint to: /content/mu-shroom-hi-xlmr-large/best


#Finding Best Params - will remove once found and replace parameters in the model configuration

In [9]:
#Performing dev param_search set RUN_DEV_PARAM_SEARCH=True once, then lock in best values.
RUN_DEV_PARAM_SEARCH = True ##set this to false if training has not been done

if RUN_DEV_PARAM_SEARCH:
    import itertools

    dev_reference_jsonl = os.path.join(PREDICTIONS_DIR, "reference_es_dev.jsonl")
    build_reference_jsonl(dev_raw, schema, dev_reference_jsonl)

    thresholds = [0.40, 0.45, 0.50, 0.55, 0.60, 0.65]
    merge_gaps = [0, 1, 2, 3]
    min_span_lens = [1, 2, 3]

    best_iou = -1
    best_params = {}
    results = []

    for threshold, merge_gap, min_span_len in itertools.product(thresholds, merge_gaps, min_span_lens):
        rows, _ = predict_dataset_to_official_rows(
            checkpoint_dir=SPANISH_CHECKPOINT_DIR,
            dataset_split=dev_raw,
            schema_map=schema,
            threshold=threshold,
            merge_gap=merge_gap,
        )
        tmp_pred = os.path.join(PREDICTIONS_DIR, "tmp_search_pred.jsonl")
        write_jsonl(tmp_pred, rows)

        iou, corr = score_predictions_with_official_scorer(dev_reference_jsonl, tmp_pred, "/dev/null")
        results.append({"threshold": threshold, "merge_gap": merge_gap, "min_span_len": min_span_len, "iou": iou, "corr": corr})

        if iou > best_iou:
            best_iou = iou
            best_params = {"threshold": threshold, "merge_gap": merge_gap, "min_span_len": min_span_len}

        print(f"thr={threshold} gap={merge_gap} min_len={min_span_len} → IoU={iou:.4f} corr={corr:.4f}")

    print("\n── Best params ──")
    print(best_params)
    print(f"Best dev IoU: {best_iou:.4f}")
    display(pd.DataFrame(results).sort_values("iou", ascending=False).head(10))

    # Lock in
    PREDICTION_THRESHOLD = best_params["threshold"]
    MERGE_GAP = best_params["merge_gap"]
    MIN_SPAN_LEN = best_params["min_span_len"]
    print(f"\nLocked: threshold={PREDICTION_THRESHOLD}, merge_gap={MERGE_GAP}, min_span_len={MIN_SPAN_LEN}")
else:
    print(f"Using params: threshold={PREDICTION_THRESHOLD}, merge_gap={MERGE_GAP}, min_span_len={MIN_SPAN_LEN}")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

thr=0.4 gap=0 min_len=1 → IoU=0.2350 corr=0.2269


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Using existing directory: /content/mu-shroom
thr=0.4 gap=0 min_len=2 → IoU=0.2350 corr=0.2269


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

...

Using existing directory: /content/mu-shroom
thr=0.65 gap=3 min_len=2 → IoU=0.2121 corr=0.2269


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Using existing directory: /content/mu-shroom
thr=0.65 gap=3 min_len=3 → IoU=0.2121 corr=0.2269

── Best params ──
{'threshold': 0.4, 'merge_gap': 1, 'min_span_len': 1}
Best dev IoU: 0.2356


,threshold,merge_gap,min_span_len,iou,corr
3,0.40,1,1,0.235629,0.226866
6,0.40,2,1,0.235629,0.226866
5,0.40,1,3,0.235629,0.226866
4,0.40,1,2,0.235629,0.226866
7,0.40,2,2,0.235629,0.226866
17,0.45,1,3,0.235629,0.226866
15,0.45,1,1,0.235629,0.226866
8,0.40,2,3,0.235629,0.226866
28,0.50,1,2,0.235629,0.226866
29,0.50,1,3,0.235629,0.226866



Locked: threshold=0.4, merge_gap=1, min_span_len=1


#Final Eval Reference

In [12]:
reference_jsonl = os.path.join(PREDICTIONS_DIR, "reference_es_eval.jsonl")
build_reference_jsonl(final_eval_raw, schema, reference_jsonl)
print("Reference file:", reference_jsonl)

Reference file: /content/mu-shroom-hi-xlmr-large/predictions/reference_es_eval.jsonl


#Evaluation English Checkpoint on Spanish and then Spanish Checkpoint on Spanish for final comparisons

In [16]:
en_to_es_results    = None
en_prediction_jsonl = os.path.join(PREDICTIONS_DIR, "predictions_en_on_es.jsonl")
en_scores_txt       = os.path.join(PREDICTIONS_DIR, "scores_en_on_es.txt")

required_files = ["config.json", "model.safetensors", "tokenizer.json", "tokenizer_config.json"]
missing        = [f for f in required_files if not os.path.isfile(os.path.join(ENGLISH_CHECKPOINT, f))]

if not os.path.isdir(ENGLISH_CHECKPOINT):
    print("English checkpoint not found:", ENGLISH_CHECKPOINT)
elif missing:
    print("Missing files:", missing)
else:
    en_rows, en_token_metrics = predict_dataset_to_official_rows(
        checkpoint_dir=ENGLISH_CHECKPOINT,
        dataset_split=final_eval_raw,
        schema_map=schema,
        threshold=PREDICTION_THRESHOLD,
        merge_gap=MERGE_GAP,
    )
    write_jsonl(en_prediction_jsonl, en_rows)
    en_iou, en_corr = score_predictions_with_official_scorer(reference_jsonl, en_prediction_jsonl, en_scores_txt)
    en_to_es_results = {
        "Model":                 "English-finetuned XLM-R on Spanish (EN→ES)",
        "Token F1 (hall micro)": en_token_metrics.get("token_f1_hall_micro", float("nan")),
        "Token macro F1":        en_token_metrics.get("token_f1_macro",      float("nan")),
        "Official IoU":          en_iou,
        "Official Correlation":  en_corr,
    }
    display(pd.DataFrame([en_to_es_results]))

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Using existing directory: /content/mu-shroom


,Model,Token F1 (hall micro),Token macro F1,Official IoU,Official Correlation
0,English-finetuned XLM-R on Spanish (EN→ES),0.277482,0.484869,0.261535,0.302811


In [18]:
es_to_es_results = None
es_prediction_jsonl = os.path.join(PREDICTIONS_DIR, "predictions_es_on_es.jsonl")
es_scores_txt = os.path.join(PREDICTIONS_DIR, "scores_es_on_es.txt")

if not os.path.isdir(SPANISH_CHECKPOINT_DIR):
    print("Spanish checkpoint not found:", SPANISH_CHECKPOINT_DIR)
else:
    es_rows, es_token_metrics = predict_dataset_to_official_rows(
        checkpoint_dir=SPANISH_CHECKPOINT_DIR,
        dataset_split=final_eval_raw,
        schema_map=schema,
        threshold=PREDICTION_THRESHOLD,
        merge_gap=MERGE_GAP,
    )
    write_jsonl(es_prediction_jsonl, es_rows)
    es_iou, es_corr = score_predictions_with_official_scorer(reference_jsonl, es_prediction_jsonl, es_scores_txt)
    es_to_es_results = {
        "Model": "Spanish-finetuned XLM-R on Spanish (ES→ES)",
        "Token F1 (hall micro)": es_token_metrics.get("token_f1_hall_micro", float("nan")),
        "Token macro F1": es_token_metrics.get("token_f1_macro",      float("nan")),
        "Official IoU": es_iou,
        "Official Correlation": es_corr,
    }
    display(pd.DataFrame([es_to_es_results]))

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Using existing directory: /content/mu-shroom


,Model,Token F1 (hall micro),Token macro F1,Official IoU,Official Correlation
0,Spanish-finetuned XLM-R on Spanish (ES→ES),0.321098,0.5016,0.294111,0.298489


In [19]:
comparison_rows = [r for r in [en_to_es_results, es_to_es_results] if r is not None]

if comparison_rows:
    display(
        pd.DataFrame(comparison_rows)
        [["Model", "Token F1 (hall micro)", "Token macro F1", "Official IoU", "Official Correlation"]]
        .sort_values("Official IoU", ascending=False)
        .reset_index(drop=True)
    )
else:
    print("No results yet — run evaluation cells first.")

,Model,Token F1 (hall micro),Token macro F1,Official IoU,Official Correlation
0,Spanish-finetuned XLM-R on Spanish (ES→ES),0.321098,0.501600,0.294111,0.298489
1,English-finetuned XLM-R on Spanish (EN→ES),0.277482,0.484869,0.261535,0.302811
